### Load dataset

In [1]:
import os
import sys
import math
from tqdm import tqdm
from pathlib import Path
from typing import Any, overload
from collections import defaultdict

import torch
import wandb
import numpy as np
from torch import nn, Tensor
import matplotlib.pyplot as plt
import torch.nn.functional as F
from torch.optim import Optimizer, AdamW
from sklearn.model_selection import train_test_split
from torch.utils.data import TensorDataset, DataLoader
from torch.optim.lr_scheduler import LRScheduler, LambdaLR

from src import configs as cfg
from src import models, plotting, dataset

/root/repos/raidium_challenge/.venv/lib/python3.13/site-packages/timm/models/layers/__init__.py:48: FutureWarning: Importing from timm.models.layers is deprecated, please import via timm.layers
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.layers", FutureWarning)


In [2]:
path = "checkpoints/swin_MiM/lyric-energy-538/swin_ssl_epoch_20.pt"

In [3]:
chkpt = torch.load(path, weights_only=False)
train_cfg = chkpt["train_cfg"]
if "mask_rec_loss_weight" in train_cfg:
    del train_cfg["mask_rec_loss_weight"]
train_cfg = cfg.TrainingConfig(**train_cfg)
model_cfg = cfg.ModelConfig(**chkpt["model_cfg"])
model = models.mk_model_from_cfg(model_cfg)
model.load_state_dict(chkpt["model"])

<All keys matched successfully>

In [4]:
loaders = dataset.mk_ssl_loaders(train_cfg)

In [7]:
batch_dict = next(iter(loaders["train"]))
batch_dict = dataset.preprocess_batch(batch_dict)
# with torch.no_grad():
#     print("with training transforms")
#     plotting.plt_recon_imgs_matplotlib(model, batch_dict, train_cfg.transform)
#     print("without transforms")
#     plotting.plt_recon_imgs_matplotlib(model, batch_dict)

In [8]:
with torch.no_grad(), torch.autocast(cfg.DEVICE.type, torch.bfloat16):
    model_output = model(batch_dict)

In [ ]:
model_output["reconstruction"].shape


torch.Size([32, 1, 256, 256])

In [20]:
import plotly.express as px

img = model_output["reconstruction"].detach().cpu().float().numpy()
img.shape
px.imshow(img[:10, 0], facet_col=0)